## **Ollama LLMs Setup**

In [ ]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

# model_client = OllamaChatCompletionClient(model="deepseek-r1:8b")
model_client = OllamaChatCompletionClient(model="qwen3:8b", parallel_tool_calls=False)
# model_client = OllamaChatCompletionClient(model="llama3.1")

In [12]:
from autogen_core.models import UserMessage
await model_client.create(
    [UserMessage(content="Hi, How are you?", source="user")],
)

CreateResult(finish_reason='stop', content='<think>\nOkay, the user greeted me with "Hi, How are you?" I need to respond in a friendly and helpful manner. Let me start by acknowledging their greeting. I should mention that I\'m just a chatbot, so I don\'t have feelings, but I\'m here to help. Maybe add a smiley emoji to keep it light. Then, invite them to ask any questions they might have. Keep it simple and positive. Let me check if that covers everything. Yes, that should work.\n</think>\n\nHi there! 😊 I\'m just a friendly chatbot, so I don\'t have feelings, but I\'m here to help you with anything you need! How can I assist you today?', usage=RequestUsage(prompt_tokens=14, completion_tokens=145), cached=False, logprobs=None, thought=None)

# ***Swarm Team***
- overall work flow on `HandoffMessages`

![Swarm](swarm.png "Swarm Agent")

## **Custom Tool**

In [3]:
def refund_flight(flight_PNR: str) -> str:
    return f"Refunded Flight with PNR {flight_PNR}"

In [4]:
from typing import List, Dict, Any
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, HandoffTermination
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console

## **Defining The Agent**

In [5]:
travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete or refund is successfully done.""",
)

flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight reference numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)


## **Termination Condition**

In [6]:
termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE")
team = Swarm([travel_agent, flights_refunder], termination_condition=termination)


# ***Run this team***

In [7]:
from autogen_agentchat.messages import HandoffMessage
task = "I need to refund my flight."


async def run_team_stream() -> None:
    task_result = await Console(team.run_stream(task=task))
    last_message = task_result.messages[-1]

    while isinstance(last_message, HandoffMessage) and last_message.target == "user":
        user_message = input("User: ")

        task_result = await Console(
            team.run_stream(task=HandoffMessage(source="user", target=last_message.source, content=user_message))
        )
        last_message = task_result.messages[-1]


# Use asyncio.run(...) if you are running this in a script.
await run_team_stream()
await model_client.close()


---------- TextMessage (user) ----------


I need to refund my flight.
---------- TextMessage (travel_agent) ----------
<think>
Okay, the user wants to refund their flight. Let me check what I need to do. The user mentioned they need to refund a flight, so the first step is to gather the necessary information. But wait, the user hasn't provided any details yet. I should ask them for the flight details like the booking reference, flight number, date of travel, and the reason for the refund. Also, I need to confirm if they have any specific requirements or if they need assistance with the refund process. Since I'm a travel agent, I should make sure to collect all the required information before handing them over to the flights_refunder. Let me structure the response to ask for these details. Once I have all the info, I can transfer to the flights_refunder function. If the user hasn't provided enough details, I should prompt them again. Alright, let me draft the message to collect the necessary information.
</think>

To process yo

In [8]:
team.reset()

<coroutine object BaseGroupChat.reset at 0x000001C83F12FB40>

## **Supervisor Agent**

![Swarm](supervisor.png "Swarm Agent")

In [9]:
async def get_stock_data(symbol: str) -> Dict[str, Any]:
    """Get stock market data for a given symbol"""
    return {"price": 180.25, "volume": 1000000, "pe_ratio": 65.4, "market_cap": "700B"}


async def get_news(query: str) -> List[Dict[str, str]]:
    """Get recent news articles about a company"""
    return [
        {
            "title": "Tesla Expands Cybertruck Production",
            "date": "2024-03-20",
            "summary": "Tesla ramps up Cybertruck manufacturing capacity at Gigafactory Texas, aiming to meet strong demand.",
        },
        {
            "title": "Tesla FSD Beta Shows Promise",
            "date": "2024-03-19",
            "summary": "Latest Full Self-Driving beta demonstrates significant improvements in urban navigation and safety features.",
        },
        {
            "title": "Model Y Dominates Global EV Sales",
            "date": "2024-03-18",
            "summary": "Tesla's Model Y becomes best-selling electric vehicle worldwide, capturing significant market share.",
        },
    ]


In [ ]:
planner = AssistantAgent(
    "planner",
    model_client=model_client,
    handoffs=["financial_analyst", "news_analyst", "writer"],
    system_message="""You are a research planning coordinator.
    Coordinate market research by delegating to specialized agents:
    - Financial Analyst: For stock data analysis
    - News Analyst: For news gathering and analysis
    - Writer: For compiling final report
    Always send your plan first, then handoff to appropriate agent.
    Always handoff to a single agent at a time.
    Use TERMINATE when research is complete.""",
)

financial_analyst = AssistantAgent(
    "financial_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_stock_data],
    system_message="""You are a financial analyst.
    Analyze stock market data using the get_stock_data tool.
    Provide insights on financial metrics.
    Always handoff back to planner when analysis is complete.""",
)

news_analyst = AssistantAgent(
    "news_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_news],
    system_message="""You are a news analyst.
    Gather and analyze relevant news using the get_news tool.
    Summarize key market insights from news.
    Always handoff back to planner when analysis is complete.""",
)

writer = AssistantAgent(
    "writer",
    model_client=model_client,
    handoffs=["planner"],
    system_message="""You are a financial report writer.
    Compile research findings into clear, concise reports.
    Always handoff back to planner when writing is complete.""",
)


In [11]:
# Define termination condition
text_termination = TextMentionTermination("TERMINATE")
termination = text_termination

research_team = Swarm(
    participants=[planner, financial_analyst, news_analyst, writer], termination_condition=termination
)

task = "Conduct market research for TSLA stock"
await Console(research_team.run_stream(task=task))
await model_client.close()


---------- TextMessage (user) ----------
Conduct market research for TSLA stock
---------- ThoughtEvent (planner) ----------
<think>
Okay, the user wants to conduct market research for TSLA stock. Let me break down how to approach this. First, I need to gather relevant information. Since the user mentioned TSLA, which is a stock, the Financial Analyst would be the right person to handle the stock data analysis. They can look into things like stock prices, trading volume, and maybe some financial metrics. 

But wait, market research isn't just about the stock data. There might be recent news affecting Tesla's stock. So after handling the financial data, I should also involve the News Analyst to gather any recent news articles or press releases that could impact the stock. They can check for things like product launches, partnerships, or any regulatory changes.

Once I have both the financial data and news analysis, the Writer can compile everything into a comprehensive report. But I nee